# Phishing Email Detector

A machine learning classifier that detects phishing emails by combining **behavioral indicator features** (urgency/authority keyword counts, link counts) with **TF-IDF text features**, trained on a combined dataset of legitimate and phishing emails.

**Dataset:** [Phishing Email Dataset](https://www.kaggle.com/datasets/naserabdullahalam/phishing-email-dataset) (Kaggle, by Naser Abdullah Alam) — this notebook uses the `CEAS_08.csv` and `Enron.csv` subsets.

**Final result:** 97.3% accuracy (Logistic Regression, TF-IDF + behavioral features)


## 1. Setup — imports

In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.dummy import DummyClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import hstack


## 2. Load the data

If running in Google Colab, use the cell below to upload `CEAS_08.csv` and `Enron.csv` (download them from the Kaggle link above first). If running locally, place both files in the same folder as this notebook and skip the upload cell.

In [ ]:
# Colab only — uncomment if uploading manually
# from google.colab import files
# uploaded = files.upload()  # select CEAS_08.csv and Enron.csv


In [ ]:
ceas = pd.read_csv('CEAS_08.csv')
enron = pd.read_csv('Enron.csv')

print("CEAS_08 labels:", ceas['label'].value_counts().to_dict())
print("Enron labels:", enron['label'].value_counts().to_dict())

df = pd.concat([ceas, enron], ignore_index=True)
print("Combined shape:", df.shape)
print("Combined labels:", df['label'].value_counts().to_dict())


## 3. Feature engineering — behavioral indicators

Three hand-crafted features meant to capture social-engineering tactics:
- `urgency_score` — count of urgency/pressure phrases
- `authority_score` — count of words implying institutional authority
- `url_count` — number of links in the email body (clipped at 10 to limit outlier influence)

**Note on the keyword list:** an earlier version included generic words like `now`, `immediately`, and `verify`. Per-word frequency analysis showed these appeared *more* often in legitimate Enron business email than in phishing email (ordinary office language, not a phishing signal), so they were removed in favor of more phishing-specific phrases.

In [ ]:
df['subject'] = df['subject'].fillna('')
df['body'] = df['body'].fillna('')
df['email_text'] = df['subject'] + ' ' + df['body']

urgency_words = ['urgent', 'limited time', 'suspended', 'click here',
                  'confirm your account', 'unusual activity', 'act fast']
authority_words = ['bank', 'official', 'government', 'security team']

def count_keywords(text, keyword_list):
    text = str(text).lower()
    return sum(1 for word in keyword_list if word in text)

df['urgency_score'] = df['email_text'].apply(lambda x: count_keywords(x, urgency_words))
df['authority_score'] = df['email_text'].apply(lambda x: count_keywords(x, authority_words))
df['url_count'] = df['body'].str.count('http').clip(upper=10)

print(df[df['label'] == 1][['urgency_score', 'authority_score', 'url_count']].mean())
print(df[df['label'] == 0][['urgency_score', 'authority_score', 'url_count']].mean())


## 4. Baseline model — behavioral features only

*Kept intentionally to show the full process.* This first attempt trains Logistic Regression on the 3 behavioral features alone, with no text features.

In [ ]:
X_baseline = df[['urgency_score', 'authority_score', 'url_count']]
y = df['label']

X_train_b, X_test_b, y_train_b, y_test_b = train_test_split(
    X_baseline, y, test_size=0.2, random_state=42)

baseline_model = LogisticRegression()
baseline_model.fit(X_train_b, y_train_b)

y_pred_b = baseline_model.predict(X_test_b)
print("Baseline accuracy:", accuracy_score(y_test_b, y_pred_b))
print(classification_report(y_test_b, y_pred_b))


**Result:** ~49.5% accuracy — below even a majority-class baseline. Confirmed with a `DummyClassifier` and by inspecting the model's coefficients (below). This shows that 3 sparse, hand-picked keyword-count features carry correct-direction but insufficient signal on their own.

In [ ]:
dummy = DummyClassifier(strategy='most_frequent')
dummy.fit(X_train_b, y_train_b)
print("Dummy baseline accuracy:", dummy.score(X_test_b, y_test_b))
print("Baseline model coefficients:", baseline_model.coef_)


## 5. Improved model — TF-IDF + behavioral features

Adding TF-IDF text features (top 1000 informative words across all emails) alongside the 3 behavioral features.

In [ ]:
tfidf = TfidfVectorizer(max_features=1000, stop_words='english')
X_text = tfidf.fit_transform(df['email_text'])

X_behavioral = df[['urgency_score', 'authority_score', 'url_count']].values
X_combined = hstack([X_text, X_behavioral])

X_train, X_test, y_train, y_test = train_test_split(
    X_combined, y, test_size=0.2, random_state=42, stratify=y)

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))


**Result:** 97.3% accuracy, with 0.97-0.98 precision/recall on both classes — a substantial improvement, showing that broad lexical signal (TF-IDF) captures phishing patterns far more effectively than a small, manually curated keyword set alone.

## 6. Manual testing on hand-written examples

Testing the trained model on original, hand-written sample emails to check real-world behavior beyond the held-out test set.

In [ ]:
def predict_samples(sample_emails):
    sample_urgency = [count_keywords(email, urgency_words) for email in sample_emails]
    sample_authority = [count_keywords(email, authority_words) for email in sample_emails]
    sample_urls = [min(email.lower().count('http'), 10) for email in sample_emails]

    sample_tfidf = tfidf.transform(sample_emails)  # transform, NOT fit_transform — reuse training vocabulary
    sample_behavioral = np.array(list(zip(sample_urgency, sample_authority, sample_urls)))
    sample_combined = hstack([sample_tfidf, sample_behavioral])

    predictions = model.predict(sample_combined)
    probabilities = model.predict_proba(sample_combined)

    for email, pred, prob in zip(sample_emails, predictions, probabilities):
        label = "PHISHING" if pred == 1 else "LEGIT"
        print(f"Prediction: {label} (confidence: {prob[pred]:.2%})")
        print(f"Email: {email[:80]}...")
        print()


In [ ]:
sample_emails_1 = [
    "URGENT: Your account has been suspended. Click here immediately to confirm your account and verify your identity or it will be permanently deleted.",
    "Hi team, just a reminder that the quarterly report is due by Friday. Let me know if you have any questions.",
    "Dear Customer, we detected unusual activity on your bank account. Please click here now to verify your information and restore access.",
    "Congratulations! We have a new job opportunity for you. Click the link above to check the details and pay for the recommendation.",
    "Pay for the ordered product while it's on the way, for smooth delivery.",
    "Congratulations! you've earned a lottery ticket for disney land, claim your tickets now by clicking this link!",
    "Hey, are we still on for lunch tomorrow? Let me know what time works for you."
]
predict_samples(sample_emails_1)


In [ ]:
sample_emails_2 = [
    "ok see you at 5",
    "Can you send me the file when you get a chance?",
    "Happy birthday! Hope you have a great day.",
    "Your Netflix subscription payment failed. Update your billing info now to avoid service interruption.",
    "Reminder: team meeting moved to 3pm today.",
    "We noticed a login attempt from a new device. If this wasn't you, secure your account immediately.",
    "lol did you see that meme I sent",
    "Thanks for your help earlier, really appreciate it!"
]
predict_samples(sample_emails_2)


## 7. Findings from manual testing

- **High-confidence predictions (>80%) were consistently accurate** — both classic phishing templates (account suspension, bank alerts) and modern ones (Netflix billing failure, login-attempt alerts) were caught correctly, despite the training data being roughly 15 years old.
- **Confidence dropped sharply (52-60%) on short, casual conversational text** ("ok see you at 5", "lol did you see that meme I sent") — likely because this register is underrepresented in the corporate/scam-heavy training data.
- **One false positive stood out:** "Happy birthday! Hope you have a great day." was misclassified as phishing at 60.45% confidence — plausibly because exclamatory, celebratory phrasing overlaps with scam templates like fake prize announcements.
- **Practical implication:** prediction confidence is itself a useful signal — predictions above ~80% were reliable in testing, while scores in the 50-65% range were not. A production system could route low-confidence predictions to human review rather than trusting all outputs equally.


## 8. Limitations

- Behavioral keyword features, while directionally correct, are sparse (most emails score 0) and insufficient as standalone predictors.
- The `url_count` feature is a simple substring count (`"http"`) and can be skewed by non-phishing link-dense content (e.g. an automated SVN commit-notification email in the dataset scored 468 before clipping).
- Only 2 of the 7 available source files in the full Kaggle dataset were used (CEAS_08, Enron); results may not generalize to the other subsets (Nazario, Nigerian_Fraud, SpamAssassin, Ling).
- Only one model type (Logistic Regression) was evaluated; no comparison against other classifiers.
- Manual testing showed reduced reliability on casual/conversational text not well represented in the training data.
